# GAT with Status Embedding — Next-Event Prediction on Helpdesk

This notebook demonstrates **next-event prediction** using the Dual-GAT architecture
enhanced with **edge-type (status) embeddings** on the Helpdesk event log.

## Model overview

| Component | Description |
|-----------|-------------|
| Architecture | Dual-GAT with an additional `nn.Embedding` layer for **edge types** |
| Edge types | Consecutive event transitions (e.g. `"Assign→Resolve"`) are label-encoded and embedded into a learned vector |
| Edge attributes | `[time_diff, edge_type_embedding]` — richer than the basic model |
| Task | Predict the **next event** at every position in the sequence |

Compared to `gat_basic`, this model gives the attention mechanism access to **transition semantics**,
allowing it to learn that certain activity transitions carry more predictive signal.

> **Note:** The Helpdesk log has only `"complete"` as lifecycle status, so edge-type
> diversity is limited here. See the BPI 2012 variant for a richer example.

### Notebook outline
1. Load & Explore the Dataset
2. Train the Model
3. Evaluate with Comprehensive Metrics
4. Summary

In [1]:
from pathlib import Path
import sys, warnings

root = Path.cwd().resolve()
src = root / "src" if (root / "src").exists() else root.parent / "src"
sys.path.insert(0, str(src))
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Load & Explore the Dataset

In [2]:
import pandas as pd

data_dir = Path.cwd() / "data" if (Path.cwd() / "data").exists() else Path.cwd() / "examples" / "data"
df = pd.read_csv(data_dir / "Helpdesk.csv")

df = df.rename(columns={
    "case:concept:name": "case_id",
    "concept:name":      "event",
    "time:timestamp":    "time",
    "lifecycle:transition": "status",
    "case:variant-index":   "variant_index",
})
df = df.dropna(subset=["case_id", "event", "time"]).copy()
df["time"] = pd.to_datetime(df["time"], utc=True)
df = df[df.groupby("case_id")["case_id"].transform("size") >= 2].reset_index(drop=True)

n_cases  = df["case_id"].nunique()
n_events = len(df)
n_status = df["status"].nunique()
print(f"Cases: {n_cases}  |  Events: {n_events}  |  Unique statuses: {n_status}")
print(f"Status values: {df['status'].unique().tolist()}")
df.head()

Cases: 4580  |  Events: 21348  |  Unique statuses: 1
Status values: ['complete']


,event,status,org:resource,time,Activity,Resource,case_id,case:variant,variant_index,case:creator
0,Assign seriousness,complete,Value 2,2010-01-13 08:40:25+00:00,Assign seriousness,Value 2,Case3608,Variant 33,33,Fluxicon Disco
1,Take in charge ticket,complete,Value 2,2010-01-29 08:52:27+00:00,Take in charge ticket,Value 2,Case3608,Variant 33,33,Fluxicon Disco
2,Resolve ticket,complete,Value 2,2010-01-29 08:52:34+00:00,Resolve ticket,Value 2,Case3608,Variant 33,33,Fluxicon Disco
3,Closed,complete,Value 5,2010-02-13 08:52:48+00:00,Closed,Value 5,Case3608,Variant 33,33,Fluxicon Disco
4,Closed,complete,Value 5,2010-02-13 08:52:48+00:00,Closed,Value 5,Case3608,Variant 33,33,Fluxicon Disco


## 2. Train the Model

The `gat_status` API requires an additional `status_col` parameter.
The library uses it to build label-encoded edge types from consecutive event+status transitions.

In [3]:
from timegnn import gat_status

result = gat_status(
    df,
    case_col="case_id",
    event_col="event",
    time_col="time",
    status_col="status",           # ← lifecycle transition column
    cat_event=["event"],
    num_event=[],
    seq_cols=["variant_index"],
    cat_seq=[],
    num_seq=["variant_index"],
    num_epochs=3,
    batch_size=16,
)

model = result["model"]
pd.DataFrame(result["history"])

,epoch,train_loss,train_acc,test_loss,test_acc
0,1,0.567640,0.835499,0.586283,0.786969
1,2,0.496190,0.843689,0.645365,0.783856
2,3,0.496539,0.841434,0.620677,0.786969


## 3. Evaluate with Comprehensive Metrics

In [4]:
import torch
from torch.utils.data import DataLoader
from timegnn import (
    EventLogTransformer, top_k_accuracy, predict, predict_per_sequence,
    average_bleu_score, compute_dls_and_exact_match, analyze_sequence_errors,
)
from timegnn.data.pyg import CustomDataset, custom_collate_fn

transformer = EventLogTransformer(
    case_col="case_id", event_col="event", time_col="time",
    status_col="status",
    cat_event=["event"], num_event=[],
    seq_cols=["variant_index"], cat_seq=[], num_seq=["variant_index"],
    mode="gat_status",
).fit(df)

transformed = transformer.transform(df)
dataset = CustomDataset(transformed.event_features, transformed.labels)
loader  = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=custom_collate_fn)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)

preds_flat, labels_flat, outputs = predict(model, loader, device)
print(f"Top-1 accuracy: {top_k_accuracy(outputs, labels_flat, k=1):.4f}")
print(f"Top-3 accuracy: {top_k_accuracy(outputs, labels_flat, k=3):.4f}")
print(f"Top-5 accuracy: {top_k_accuracy(outputs, labels_flat, k=5):.4f}")

preds_seq, labels_seq = predict_per_sequence(model, loader, device)
bleu = average_bleu_score(preds_seq, labels_seq)
dls, exact = compute_dls_and_exact_match(preds_seq, labels_seq)
print(f"\nBLEU:  {bleu:.4f}")
print(f"DLS:   {dls:.4f}")
print(f"Exact: {exact:.4f}")

print("\nSequence error summary:")
print(analyze_sequence_errors(model, loader, device))

Top-1 accuracy: 0.8321
Top-3 accuracy: 0.9819
Top-5 accuracy: 0.9966

BLEU:  0.6746
DLS:   0.8611
Exact: 0.5400

Sequence error summary:
(array([ 744, 1325,  793,  324,  229,   78,   44,   19,   11,    9,    5,
          2,    1,    1]), {(10, 12): 584, (10, 8): 70, (12, 0): 405, (10, 14): 1234, (12, 10): 276, (1, 10): 234, (10, 2): 28, (14, 2): 28, (1, 12): 120, (14, 8): 31, (14, 10): 129, (14, 11): 5, (10, 1): 65, (14, 1): 18, (12, 14): 61, (0, 14): 12, (10, 4): 7, (12, 1): 33, (4, 1): 16, (0, 12): 79, (1, 7): 1, (13, 1): 2, (3, 10): 8, (14, 12): 68, (12, 2): 6, (9, 8): 2, (12, 8): 8, (4, 12): 2, (4, 10): 2, (3, 8): 1, (3, 4): 1, (1, 8): 1, (1, 14): 3, (0, 10): 8, (1, 4): 8, (14, 0): 4, (0, 8): 1, (9, 10): 4, (7, 1): 1, (2, 9): 3, (2, 10): 2, (12, 4): 1, (14, 4): 2, (14, 9): 1, (8, 4): 1, (8, 10): 1, (2, 8): 2, (3, 1): 1, (8, 2): 1, (7, 4): 1, (10, 0): 1, (10, 13): 1, (10, 9): 1}, {'min': 2, 'max': 15, 'mean': np.float64(4.661135371179039), 'median': np.float64(4.0)})


## 4. Summary

The **status embedding** model enriches edge representations with learned transition embeddings.
In this dataset all events share the same lifecycle status (`complete`), so the benefit is limited.

For a dataset with diverse lifecycle transitions (e.g. `SCHEDULE`, `START`, `COMPLETE`),
see **`gat_status_bpi_full`**.